In [1]:
# Cell 1 — Install
!pip -q install -U langgraph langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 8.1 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports and state

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command


class AgentState(TypedDict):
    user_input: str
    classification: str
    response: str


In [3]:
# Cell 3 — Node 1: classify

def classify(state: AgentState):
    text = state["user_input"].lower()

    if any(word in text for word in ["refund", "payment", "charge", "invoice"]):
        category = "billing"
    elif any(word in text for word in ["error", "bug", "crash", "not working"]):
        category = "technical"
    elif any(word in text for word in ["hello", "hi", "hey"]):
        category = "general"
    else:
        category = "human_review"

    return {"classification": category}


In [4]:
# Cell 4 — Node 2: route + Human-in-the-loop interrupt

def route(state: AgentState):
    category = state["classification"]

    # Pause for human review when classification is uncertain
    if category == "human_review":
        human_decision = interrupt({
            "message": "Human review required.",
            "user_input": state["user_input"],
            "question": "Classify this request as billing, technical, or general."
        })

        category = human_decision

    return {"classification": category}


In [5]:
# Cell 5 — Node 3: respond

def respond(state: AgentState):
    category = state["classification"]

    responses = {
        "billing": "Billing team: I can help with payments, refunds, or invoices.",
        "technical": "Technical support: I can help troubleshoot the issue.",
        "general": "General support: How can I help you today?"
    }

    return {
        "response": responses.get(
            category,
            "Sorry, I could not determine the correct department."
        )
    }


In [6]:
# Cell 6 — Conditional routing

def route_condition(state: AgentState):
    return state["classification"]


In [7]:
# Cell 7 — Build LangGraph

graph_builder = StateGraph(AgentState)

graph_builder.add_node("classify", classify)
graph_builder.add_node("route", route)
graph_builder.add_node("respond", respond)

graph_builder.add_edge(START, "classify")

# classify -> route
graph_builder.add_conditional_edges(
    "classify",
    route_condition,
    {
        "billing": "route",
        "technical": "route",
        "general": "route",
        "human_review": "route",
    }
)

# route -> respond
graph_builder.add_edge("route", "respond")
graph_builder.add_edge("respond", END)

graph = graph_builder.compile()


In [ ]:
# Cell 7 — Build LangGraph

graph_builder = StateGraph(AgentState)

graph_builder.add_node("classify", classify)
graph_builder.add_node("route", route)
graph_builder.add_node("respond", respond)

graph_builder.add_edge(START, "classify")

# classify -> route
graph_builder.add_conditional_edges(
    "classify",
    route_condition,
    {
        "billing": "route",
        "technical": "route",
        "general": "route",
        "human_review": "route",
    }
)

# route -> respond
graph_builder.add_edge("route", "respond")
graph_builder.add_edge("respond", END)

graph = graph_builder.compile()


In [8]:
# Cell 9 — Human-in-the-loop test
# The last input triggers interrupt().

from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

graph_with_memory = graph_builder.compile(
    checkpointer=checkpointer
)

config = {
    "configurable": {
        "thread_id": "demo-thread-1"
    }
}

result = graph_with_memory.invoke(
    {
        "user_input": "I have a problem that needs special attention",
        "classification": "",
        "response": ""
    },
    config=config
)

print("Graph paused for human input.")
print(result)


Graph paused for human input.
{'user_input': 'I have a problem that needs special attention', 'classification': 'human_review', 'response': '', '__interrupt__': [Interrupt(value={'message': 'Human review required.', 'user_input': 'I have a problem that needs special attention', 'question': 'Classify this request as billing, technical, or general.'}, id='5a562b5611d1a3e33a02fbd822b26577')]}


In [9]:
# Cell 10 — Resume after human decision

# Human decides that this request belongs to technical support.
resumed = graph_with_memory.invoke(
    Command(resume="technical"),
    config=config
)

print("Classification:", resumed["classification"])
print("Response:", resumed["response"])


Classification: technical
Response: Technical support: I can help troubleshoot the issue.


In [10]:
# Cell 11 — Simple routing verification

expected = {
    "I want a refund for my payment": "billing",
    "The application crashes when I login": "technical",
    "Hi, I need some help": "general",
    "My invoice has an incorrect charge": "billing",
}

for text, expected_category in expected.items():
    result = graph.invoke({
        "user_input": text,
        "classification": "",
        "response": ""
    })

    assert result["classification"] == expected_category

print("✅ All routing tests passed!")


✅ All routing tests passed!
